# Personal Intelligence System — Google Colab Runner

This notebook runs the **Personal Intelligence System** directly inside Google Colab with **zero Docker containers**.
- **Database**: SQLite (persisted to Google Drive)
- **Vector Store**: Qdrant Embedded on-disk (persisted to Google Drive)
- **Reranker**: GPU-accelerated Cross-Encoder (`sentence-transformers`)
- **LLM**: OpenRouter API

### Step 1: Mount Google Drive (for persistent data)

In [ ]:
import os
import sys

# Set to True to persist SQLite and Qdrant vector database in Google Drive:
MOUNT_GOOGLE_DRIVE = False

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/personal-intelligence'
else:
    if os.path.exists('/content/personal-intelligence'):
        PROJECT_DIR = '/content/personal-intelligence'
    elif os.path.exists('./personal_intelligence'):
        PROJECT_DIR = os.getcwd()
    else:
        PROJECT_DIR = '/content/personal-intelligence'

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
%cd {PROJECT_DIR}
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print(f'Working in: {PROJECT_DIR}')


### Step 2: Install Dependencies
*Tip: Select `Runtime > Change runtime type > T4 GPU` for fastest reranking.*

In [ ]:
!pip install -q \
    "openai>=1.30.0" \
    "qdrant-client>=1.9.0" \
    "sqlalchemy[asyncio]>=2.0.0" \
    "aiosqlite>=0.20.0" \
    "greenlet>=3.0.0" \
    "sentence-transformers>=3.0.0" \
    "rank-bm25>=0.2.2" \
    "tiktoken>=0.7.0" \
    "pydantic>=2.7.0" \
    "pydantic-settings>=2.2.0" \
    "structlog>=24.0.0" \
    "tenacity>=8.3.0" \
    "prometheus-client>=0.20.0" \
    "fastapi>=0.111.0" \
    "uvicorn[standard]>=0.29.0" \
    "rich>=13.7.0"

print("Dependencies installed successfully!")


### Step 3: Configure Environment Variables

In [ ]:
import os

# OpenRouter API Key & Free Models
# Tip: In Google Colab, you can store your key in the 'Secrets' (🔑) tab as OPENROUTER_API_KEY
try:
    from google.colab import userdata
    api_key = userdata.get('OPENROUTER_API_KEY')
except Exception:
    api_key = None

if not api_key:
    api_key = os.environ.get('OPENROUTER_API_KEY')

if not api_key or api_key == 'your-openrouter-key-here':
    # Enter your OpenRouter key here if not using Colab Secrets:
    api_key = 'your-openrouter-key-here'

os.environ['OPENROUTER_API_KEY'] = api_key
os.environ['OPENROUTER_DEFAULT_MODEL'] = 'nvidia/nemotron-3-super-120b-a12b:free'
os.environ['OPENROUTER_FAST_MODEL'] = 'liquid/lfm-2.5-2.6b:free'
os.environ['OPENROUTER_EMBEDDING_MODEL'] = 'openai/text-embedding-3-small'

# Zero-Docker persistent database & vector store in data directory
active_project_dir = globals().get('PROJECT_DIR', os.getcwd())
os.environ['DATABASE_URL'] = f'sqlite+aiosqlite:///{active_project_dir}/data/personal_intelligence.db'
os.environ['QDRANT_PATH'] = f'{active_project_dir}/data/qdrant'
os.environ['NEO4J_ENABLED'] = 'false'  # Retriever uses Vector + BM25 without requiring Neo4j

print('Environment configured with OpenRouter free models!')


### Step 4: Initialize Database Schema

In [ ]:
import os
import sys

active_project_dir = globals().get('PROJECT_DIR', os.getcwd())
possible_paths = [
    active_project_dir,
    os.path.join(active_project_dir, 'personal-intelligence'),
    '/content/personal-intelligence',
    '/content',
    os.getcwd(),
]

found_dir = None
for p in possible_paths:
    if os.path.isdir(os.path.join(p, 'personal_intelligence')):
        found_dir = p
        break

if found_dir:
    print(f'Found project files in: {found_dir}')
    if found_dir not in sys.path:
        sys.path.insert(0, found_dir)
    from personal_intelligence.infrastructure.db.migrate import main as migrate_db
    await migrate_db()
else:
    print('Error: personal_intelligence directory not found.')
    print('Please ensure personal-intelligence files are in your working directory.')


### Step 5: Test Ingestion and Hybrid Retrieval

In [ ]:
from personal_intelligence.core.llm.client import llm_client
from personal_intelligence.core.memory.vector_store import vector_store
from personal_intelligence.core.graph.client import graph_client
from personal_intelligence.core.rag.ingest import ingest_text
from personal_intelligence.core.rag.retriever import HybridRetriever
from personal_intelligence.infrastructure.db.repositories import ChunkRepository

# 1. Connect services
await vector_store.connect()
await graph_client.connect()

# 2. Test LLM completion with free model
llm_test = await llm_client.complete([{'role': 'user', 'content': 'Confirm you are active in 5 words or less.'}])
print(f'LLM Status ({llm_test.model}): {llm_test.content.strip()}')

# 3. Ingest some test notes
repo = ChunkRepository()
sample_doc = """
Personal Intelligence System is running smoothly in Google Colab.
It uses SQLite for relational chunk storage, embedded Qdrant for semantic vector search,
and a GPU-accelerated CrossEncoder for precision reranking.
OpenRouter handles LLM inference via free models.
"""
doc_id = await ingest_text(sample_doc, source='colab_test.md', repo=repo)
print(f'\nDocument ingested with doc_id: {doc_id}')

# 4. Query the retriever
query = 'How are vectors and embeddings stored?'
retriever = HybridRetriever(repo)
results = await retriever.retrieve(query, top_k=3)

print(f'\nRetrieved {len(results)} results for query: \'{query}\':')
for r in results:
    print(f'- [Score: {r.score:.3f} | {r.retrieval_method}] {r.text.strip()}')

# 5. Check session costs
print('\nSession Cost Summary:', llm_client.session_cost_summary())


### Step 6 (Optional): Expose FastAPI via Cloudflare Tunnel

In [ ]:
# Download cloudflared inside Colab
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

# Run Cloudflare Tunnel (creates a public https://*.trycloudflare.com URL)
!cloudflared tunnel --url http://localhost:8000